In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import importlib

import utils

results_dir = Path() / "results" / "sphere_tracing_loop_structure"
figures_dir = Path() / "figures"

In [ ]:
cache = True # Set to True to skip already benchmarked configurations
num_rays = [100, 1000, 10000, 100000] # , 1000000, 10000000

results_dir.mkdir(parents=True, exist_ok=True)

for num_ray in num_rays:
    for method, kwargs in [("iarap", dict(ray_split_threshold=10000)), ("ling", dict())]: # Disable splitting for `iarap`
        output_path = results_dir / f"{method}_{num_ray}.csv"
        if output_path.exists() and cache:
            print(f"Skipping {method} with {num_ray} rays (cached).")
            continue
        
        utils.call_script(Path().absolute() / "benchmark_sphere_tracing.py", method=method, num_rays=num_ray, dataset="neural_iarap", num_runs=5, 
                          **kwargs, output=output_path)

In [ ]:
importlib.reload(utils)

# Load and merge all dataframes
dataframes = [pd.read_csv(f) for f in results_dir.glob("*.csv")]
df_raw = pd.concat(dataframes, ignore_index=True)

# Compute averages and standard deviations over runs, and merge them into a single dataframe
df_avg_by_sdf = df_raw.groupby(["method", "num_rays", "dataset", "sdf_name"]).agg({"elapsed_time": "mean", "peak_memory": "mean"}).reset_index()
df_std_by_sdf = df_raw.groupby(["method", "num_rays", "dataset", "sdf_name"]).agg({"elapsed_time": "std", "peak_memory": "std"}).reset_index()
df_by_sdf = pd.merge(df_avg_by_sdf, df_std_by_sdf, on=["method", "num_rays", "dataset", "sdf_name"], suffixes=("_avg", "_std"))

# Compute averages over SDFs, and merge them into a single dataframe
df = df_by_sdf.groupby(["method", "num_rays"]).agg({"elapsed_time_avg": "mean", "peak_memory_avg": "mean"}).reset_index()

# Only use seaborn colors for the plots
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.style.library["seaborn-v0_8"]['axes.prop_cycle'])

fig = plt.figure(figsize=(6, 5))
gs = fig.add_gridspec(2, 1, hspace=0.08)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax1.tick_params(labelbottom=False)

utils.set_figure_title_with_specs(fig, benchmark_name="Sphere Tracing (Loop Structure)")
utils.plot_bar_chart(ax1, df, metric="elapsed_time", metric_label="Runtime (ms)", method_to_label={"ling": "Nested loops", "iarap": "Single loop"}, reference_method="ling", y_quantile=0.85, x_label=False)
utils.plot_bar_chart(ax2, df, metric="peak_memory", metric_label="Peak memory (MiB)", method_to_label={"ling": "Nested loops", "iarap": "Single loop"}, reference_method="ling", y_quantile=0.85, legend=False)

figures_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figures_dir / "benchmark_sphere_tracing_loop_structure.png", dpi=150, bbox_inches="tight")